# CardioAI workflow (report-aligned clean version)

This notebook is a cleaned, report-aligned version of the original working notebook used in the CardioAI project.

It keeps the core PTB-XL workflow relevant to the thesis:

- environment and path configuration
- preprocessing and beat-token preparation
- PTB-XL metadata loading and label construction
- data build/load from Drive
- main PTB-XL pipeline
- evaluation and bootstrap confidence intervals
- Integrated Gradients example code for XAI

This version intentionally removes:

- repeated mount/install/debug cells
- application demo cells
- Git/GitHub clone cells
- large saved outputs
- embedded access tokens

## Recommended use

Before running this notebook:

1. mount Google Drive in Colab
2. set dataset and output paths in the configuration cell below
3. install dependencies from the repository `requirements.txt`
4. execute the cells in order

This notebook is for research and educational use only. It is not intended for clinical deployment or medical decision-making.

In [ ]:
# ============================================================
# CELL 1 — MOUNT DRIVE + PATHS
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os

# Replace these with your actual locations in Google Drive
DRIVE_ROOT = os.environ.get("CARDIOAI_DRIVE_ROOT", "/content/drive/MyDrive")
RUN_DIR = os.environ.get("CARDIOAI_RUN_DIR", os.path.join(DRIVE_ROOT, "cardioai_logs", "ptbxl_build_20260128_173743"))
PTBXL_ROOT = os.environ.get(
    "CARDIOAI_PTBXL_ROOT",
    os.path.join(DRIVE_ROOT, "CardioAI Datasets", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
)

print("RUN_DIR:", RUN_DIR)
print("PTBXL_ROOT:", PTBXL_ROOT)
print("RUN_DIR exists:", os.path.exists(RUN_DIR))
print("PTBXL_ROOT exists:", os.path.exists(PTBXL_ROOT))

In [ ]:
# ============================================================
# CELL 2 — IMPORTS + CONFIG
# ============================================================
import ast, json, time, math, shutil, random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix
from tqdm.auto import tqdm

from scipy.signal import butter, filtfilt, iirnotch
from scipy.stats import ttest_rel

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

@dataclass
class CFG:
    # Paths
    RUN_DIR: str = RUN_DIR
    PTBXL_ROOT: str = PTBXL_ROOT

    # Task setup: "multilabel" OR "multiclass3" OR "binary"
    TASK: str = "multilabel"   # <- matches your y_multi_hot earlier
    # multilabel classes in PTB-XL diagnostic superclasses
    ML_CLASSES: Tuple[str,...] = ("NORM","MI","STTC","CD")

    # Preprocessing (report-aligned core)
    PREPROCESS: bool = True
    BANDPASS_LOW: float = 0.5
    BANDPASS_HIGH: float = 40.0
    NOTCH_HZ: float = 50.0
    NOTCH_Q: float = 30.0
    ZSCORE: bool = True

    # Optional representation
    USE_STFT: bool = False  # keep False unless you really want TF rep
    STFT_NFFT: int = 256
    STFT_HOP: int = 64

    # Training (report-aligned)
    BATCH_SIZE: int = 64
    LR: float = 1e-4
    DROPOUT: float = 0.3
    MAX_EPOCHS: int = 50
    EARLY_STOP: int = 10
    LR_FACTOR: float = 0.1
    NUM_FOLDS: int = 5

    # Evaluation
    THR: float = 0.5
    BOOTSTRAP_N: int = 1000
    PLOT: bool = True

cfg = CFG()

In [ ]:
# ============================================================
# CELL 3 — PREPROCESSING (band-pass + notch + z-score) + optional STFT
# ============================================================
def bandpass_filter(x: np.ndarray, fs: int, low: float, high: float, order: int = 4) -> np.ndarray:
    # x: (C,L)
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, x, axis=1)

def notch_filter(x: np.ndarray, fs: int, notch_hz: float, q: float = 30.0) -> np.ndarray:
    nyq = 0.5 * fs
    w0 = notch_hz / nyq
    b, a = iirnotch(w0, q)
    return filtfilt(b, a, x, axis=1)

def zscore(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True)
    return (x - mu) / (sd + eps)

def preprocess_ecg(x: np.ndarray, fs: int, cfg: CFG) -> np.ndarray:
    # x: (C,L)
    if cfg.PREPROCESS:
        x = bandpass_filter(x, fs, cfg.BANDPASS_LOW, cfg.BANDPASS_HIGH)
        x = notch_filter(x, fs, cfg.NOTCH_HZ, cfg.NOTCH_Q)
    if cfg.ZSCORE:
        x = zscore(x)
    return x.astype(np.float32)

def stft_transform(x: torch.Tensor, cfg: CFG) -> torch.Tensor:
    """
    x: (B,C,L) -> returns (B,C,F,T)
    """
    # torch.stft expects (B*C, L)
    B,C,L = x.shape
    x2 = x.reshape(B*C, L)
    X = torch.stft(
        x2,
        n_fft=cfg.STFT_NFFT,
        hop_length=cfg.STFT_HOP,
        win_length=cfg.STFT_NFFT,
        return_complex=True
    )
    X = torch.abs(X)  # magnitude
    # reshape back
    F,T = X.shape[-2], X.shape[-1]
    return X.reshape(B, C, F, T)

In [ ]:
# ============================================================
# CELL 4 — LOAD PTB-XL METADATA + BUILD LABELS (NaN-safe)
# ============================================================
import os, ast
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List

def load_ptbxl_metadata(ptbxl_root: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    db = pd.read_csv(os.path.join(ptbxl_root, "ptbxl_database.csv"))
    scp = pd.read_csv(os.path.join(ptbxl_root, "scp_statements.csv"), index_col=0)

    # Parse scp_codes safely
    db["scp_codes"] = db["scp_codes"].apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else {})

    # ---- FIX: NaN-safe numeric flags in scp_statements ----
    for col in ["diagnostic", "form", "rhythm"]:
        if col in scp.columns:
            scp[col] = pd.to_numeric(scp[col], errors="coerce").fillna(0).astype(int)

    return db, scp

def scp_to_diag_superclasses(scp_codes: Dict, scp_df: pd.DataFrame, keep: Tuple[str, ...]) -> List[str]:
    out = set()
    for code in scp_codes.keys():
        if code not in scp_df.index:
            continue

        # diagnostic flag (already sanitized to int 0/1)
        if "diagnostic" in scp_df.columns and int(scp_df.at[code, "diagnostic"]) != 1:
            continue

        cls = scp_df.at[code, "diagnostic_class"] if "diagnostic_class" in scp_df.columns else None
        if isinstance(cls, str) and cls in keep:
            out.add(cls)

    return sorted(out)

def build_targets(db: pd.DataFrame, scp: pd.DataFrame, cfg):
    # multi-label 4-class target from PTB-XL diagnostic superclasses
    supers = db["scp_codes"].apply(lambda d: scp_to_diag_superclasses(d, scp, cfg.ML_CLASSES))
    y_ml = np.zeros((len(db), len(cfg.ML_CLASSES)), dtype=np.float32)
    for i, lst in enumerate(supers):
        for c in lst:
            y_ml[i, cfg.ML_CLASSES.index(c)] = 1.0

    # derived 3-class label for stratification / multiclass option
    idx = {c: i for i, c in enumerate(cfg.ML_CLASSES)}
    norm = y_ml[:, idx["NORM"]]
    mi   = y_ml[:, idx["MI"]]
    sttc = y_ml[:, idx["STTC"]]
    cd   = y_ml[:, idx["CD"]]

    label3 = np.full(len(db), -1, dtype=np.int64)  # 0 normal, 1 ischemic, 2 arrhythmic
    ischemic = (mi + sttc) > 0
    arrh = (cd > 0) & (~ischemic)
    normal = (norm > 0) & (~ischemic) & (~arrh)

    label3[normal] = 0
    label3[ischemic] = 1
    label3[arrh] = 2

    # binary: disease vs normal
    y_bin = np.where(label3 == 0, 0, 1).astype(np.int64)

    return y_ml, label3, y_bin

def make_fold_ids(y_strat: np.ndarray, patient_ids: np.ndarray, n_splits: int, seed: int) -> np.ndarray:
    from sklearn.model_selection import StratifiedGroupKFold
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_ids = np.full(len(y_strat), -1, dtype=np.int64)
    dummyX = np.zeros((len(y_strat), 1))
    for f, (_, va) in enumerate(sgkf.split(dummyX, y_strat, groups=patient_ids)):
        fold_ids[va] = f
    assert (fold_ids >= 0).all()
    return fold_ids

In [ ]:
# ============================================================
# CELL 5 — DATA BUILD/LOAD (NO DOWNLOADS; ALWAYS USE DRIVE PTB-XL)
# ============================================================
import wfdb

def backup_existing_arrays(run_dir: str) -> Optional[str]:
    files = ["X_ptbxl_10s.npy","X_ptbxl_10s_preproc.npy","y_multi_hot.npy","patient_ids.npy","fold_ids_5fold.npy"]
    present = [f for f in files if os.path.exists(os.path.join(run_dir, f))]
    if not present:
        return None
    ts = time.strftime("%Y%m%d_%H%M%S")
    bdir = os.path.join(run_dir, f"_backup_before_rebuild_{ts}")
    os.makedirs(bdir, exist_ok=True)
    for f in present:
        shutil.copy2(os.path.join(run_dir, f), os.path.join(bdir, f))
    return bdir

def rebuild_from_raw_ptbxl(cfg: CFG):
    print("🔁 Rebuilding aligned X, y, patient_ids from raw PTB-XL in DRIVE (records500)...")

    db, scp = load_ptbxl_metadata(cfg.PTBXL_ROOT)
    y_ml, label3, y_bin = build_targets(db, scp, cfg)

    # keep only rows that map to our target space (label3 != -1)
    keep_mask = label3 != -1
    db = db.loc[keep_mask].reset_index(drop=True)
    y_ml = y_ml[keep_mask]
    label3 = label3[keep_mask]
    y_bin = y_bin[keep_mask]

    # load signals
    X = np.zeros((len(db), 12, 5000), dtype=np.float32)
    patient_ids = db["patient_id"].values.astype(np.int64)

    for i in tqdm(range(len(db)), desc="Loading records500"):
        rec = os.path.join(cfg.PTBXL_ROOT, db.loc[i, "filename_hr"])  # already includes records500/...
        sig, meta = wfdb.rdsamp(rec)  # sig: (L,12)
        fs = int(meta["fs"])
        x = sig.T  # (12,L)
        # ensure 10s @ 500Hz => 5000
        if x.shape[1] != 5000:
            # safe crop/pad
            if x.shape[1] > 5000:
                x = x[:, :5000]
            else:
                pad = 5000 - x.shape[1]
                x = np.pad(x, ((0,0),(0,pad)), mode="constant")
        x = preprocess_ecg(x, fs, cfg)
        X[i] = x

    # save arrays
    np.save(os.path.join(cfg.RUN_DIR, "X_ptbxl_10s_preproc.npy"), X)
    np.save(os.path.join(cfg.RUN_DIR, "y_multi_hot.npy"), y_ml.astype(np.float32))
    np.save(os.path.join(cfg.RUN_DIR, "patient_ids.npy"), patient_ids)

    # folds (stratify using 3-class label, group by patient)
    fold_ids = make_fold_ids(label3, patient_ids, cfg.NUM_FOLDS, SEED)
    np.save(os.path.join(cfg.RUN_DIR, "fold_ids_5fold.npy"), fold_ids)

    print("✅ Saved:")
    print(" - X_ptbxl_10s_preproc.npy:", X.shape)
    print(" - y_multi_hot.npy:", y_ml.shape)
    print(" - patient_ids.npy:", patient_ids.shape)
    print(" - fold_ids_5fold.npy:", fold_ids.shape)

def load_or_build(cfg: CFG):
    x_path = os.path.join(cfg.RUN_DIR, "X_ptbxl_10s_preproc.npy")
    y_path = os.path.join(cfg.RUN_DIR, "y_multi_hot.npy")
    p_path = os.path.join(cfg.RUN_DIR, "patient_ids.npy")
    f_path = os.path.join(cfg.RUN_DIR, "fold_ids_5fold.npy")

    if all(os.path.exists(p) for p in [x_path, y_path, p_path, f_path]):
        X = np.load(x_path)
        y_ml = np.load(y_path)
        patient_ids = np.load(p_path)
        fold_ids = np.load(f_path)
        # quick safety check
        if len(X) == len(y_ml) == len(patient_ids) == len(fold_ids):
            print("✅ Loaded aligned arrays from RUN_DIR")
            return X, y_ml, patient_ids, fold_ids

    # otherwise rebuild (NO DOWNLOAD)
    bdir = backup_existing_arrays(cfg.RUN_DIR)
    if bdir:
        print("🧷 Backed up old arrays to:", bdir)
    rebuild_from_raw_ptbxl(cfg)
    return load_or_build(cfg)

X_np, y_ml_np, patient_ids_np, fold_ids_np = load_or_build(cfg)
print("X:", X_np.shape, "| y_multi_hot:", y_ml_np.shape, "| folds:", len(np.unique(fold_ids_np)))

In [ ]:
# ============================================================
# CardioAI — MAIN PIPELINE (PTB-XL from Google Drive, NO DOWNLOAD)
# - Uses your Drive PTB-XL folder (records500 + csvs)
# - Fixes patient_ids without downloading:
#     (1) FAST align y_multi_hot to metadata -> save patient_ids.npy
#     (2) If alignment fails -> rebuild aligned X,y,patient_ids ONCE from Drive records500
# - Patient-wise 70/15/15 split + 5-fold patient-wise CV on train+val
# - Models: CNN baseline + Hybrid CNN-Transformer
# - Prints ALL metrics: Accuracy, Precision, Recall/Sensitivity, Specificity, F1, AUROC
# - Visuals (blue hues): training curves, macro-metric bars, ROC curves, per-class 2x2 confusions
# ============================================================

# =========================
# CELL 0 — Installs
# =========================
#!pip -q install numpy pandas scipy tqdm wfdb scikit-learn matplotlib

# =========================
# CELL 1 — Safe Drive mount + Paths
# =========================
import os, shutil
from google.colab import drive

MOUNT_POINT = "/content/drive"
try:
    drive.flush_and_unmount()
except Exception:
    pass

if os.path.exists(MOUNT_POINT) and os.listdir(MOUNT_POINT):
    shutil.rmtree(MOUNT_POINT)
os.makedirs(MOUNT_POINT, exist_ok=True)
drive.mount(MOUNT_POINT, force_remount=True)

# >>> YOUR RUN_DIR (logs + saved npy)
RUN_DIR = "${DRIVE_ROOT}/cardioai_logs/ptbxl_build_20260128_173743"

# >>> YOUR PTB-XL DATASET IN DRIVE (contains records500/, ptbxl_database.csv, scp_statements.csv)
PTBXL_ROOT_DRIVE = "${DRIVE_ROOT}/CardioAI Datasets/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

print("RUN_DIR exists:", os.path.exists(RUN_DIR))
print("PTBXL_ROOT_DRIVE exists:", os.path.exists(PTBXL_ROOT_DRIVE))
print("PTBXL_ROOT sample files:", os.listdir(PTBXL_ROOT_DRIVE)[:12])

assert os.path.exists(os.path.join(PTBXL_ROOT_DRIVE, "ptbxl_database.csv"))
assert os.path.exists(os.path.join(PTBXL_ROOT_DRIVE, "scp_statements.csv"))
assert os.path.exists(os.path.join(PTBXL_ROOT_DRIVE, "records500"))

# =========================
# CELL 2 — Imports
# =========================
import json, math, time, random, ast
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm

from scipy.signal import butter, filtfilt, iirnotch
from scipy.stats import ttest_rel

from sklearn.metrics import roc_auc_score, roc_curve, auc

import matplotlib.pyplot as plt

# =========================
# CELL 3 — Repro helpers
# =========================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def softmax_np(x: np.ndarray, axis=-1) -> np.ndarray:
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=axis, keepdims=True) + 1e-12)

def safe_div(a, b):
    return a / (b + 1e-12)

# =========================
# CELL 4 — Config (report-matched)
# =========================
@dataclass
class Config:
    RUN_DIR: str = RUN_DIR
    PTBXL_ROOT: str = PTBXL_ROOT_DRIVE

    FS: int = 500
    N_LEADS: int = 12
    SIG_LEN: int = 5000

    # Your saved task is MULTILABEL 4-class (NORM, MI, STTC, CD)
    TASK: str = "multilabel"
    N_CLASSES: int = 4
    CLASS_NAMES: Tuple[str, ...] = ("NORM", "MI", "STTC", "CD")

    # Preprocess (report-style)
    PRE_BANDPASS: bool = True
    BP_LOW: float = 0.5
    BP_HIGH: float = 40.0
    BP_ORDER: int = 4
    PRE_NOTCH: bool = False
    NOTCH_HZ: float = 50.0
    NOTCH_Q: float = 30.0
    PRE_ZSCORE: bool = True

    FORCE_REPROCESS: bool = False

    # Patient-wise split
    TRAIN_FRAC: float = 0.70
    VAL_FRAC: float = 0.15
    TEST_FRAC: float = 0.15
    SPLIT_SEARCH_TRIALS: int = 2000

    # CV
    FOLDS: int = 5

    # Training (report-matched)
    SEED: int = 42
    BATCH_SIZE: int = 64
    GRAD_ACCUM_STEPS: int = 1
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    DROPOUT: float = 0.3
    MAX_EPOCHS: int = 50
    EARLY_STOP_PATIENCE: int = 10
    PLATEAU_PATIENCE: int = 3
    PLATEAU_FACTOR: float = 0.1
    CLIP_GRAD_NORM: float = 1.0

    THR: float = 0.50

    # Imbalance
    USE_POS_WEIGHT: bool = True
    USE_WEIGHTED_SAMPLER: bool = False

    # CI
    CI_N_BOOT: int = 1000
    CI_ALPHA: float = 0.05
    CI_PATIENTWISE: bool = True

    NUM_WORKERS: int = 2
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    RUN_MODELS: Tuple[str, ...] = ("cnn", "hybrid")
    SAVE_BEST: bool = True

    # Hybrid params
    EMBED_DIM: int = 128
    N_HEADS: int = 4
    N_LAYERS: int = 4
    FF_DIM: int = 256
    CNN_CHANNELS: Tuple[int, ...] = (64, 128, 256)

    # Visuals
    SHOW_PLOTS: bool = True

cfg = Config()
seed_everything(cfg.SEED)
print("device:", cfg.DEVICE)

# =========================
# CELL 5 — VisualAgent (blue hues)
# =========================
class VisualAgent:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _blue_colors(self, k: int):
        cmap = plt.cm.Blues
        return [cmap(x) for x in np.linspace(0.45, 0.9, k)]

    def plot_history(self, history: Dict, title: str):
        if not self.cfg.SHOW_PLOTS: return
        plt.figure()
        plt.plot(history["train_loss"], label="train")
        plt.plot(history["val_loss"], label="val")
        plt.title(title)
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.legend()
        plt.show()

    def plot_macro_bar(self, macro_list: List[Dict], labels: List[str], title: str):
        if not self.cfg.SHOW_PLOTS: return
        keys = ["ACC","P","R","SENS","SPEC","F1","AUROC"]
        x = np.arange(len(keys))
        width = 0.9 / max(len(macro_list), 1)
        colors = self._blue_colors(len(macro_list))

        plt.figure(figsize=(10,4))
        for i, m in enumerate(macro_list):
            vals = [m.get(k, np.nan) for k in keys]
            plt.bar(x + i*width, vals, width=width, label=labels[i], color=colors[i])
        plt.xticks(x + width*(len(macro_list)-1)/2, keys)
        plt.ylim(0, 1.0)
        plt.title(title)
        plt.legend()
        plt.show()

    def plot_roc(self, y_true: np.ndarray, y_prob: np.ndarray, title: str):
        if not self.cfg.SHOW_PLOTS: return
        colors = self._blue_colors(self.cfg.N_CLASSES)
        plt.figure(figsize=(6,5))
        for i, cls in enumerate(self.cfg.CLASS_NAMES):
            yt = y_true[:, i]
            yp = y_prob[:, i]
            if len(np.unique(yt)) < 2:
                continue
            fpr, tpr, _ = roc_curve(yt, yp)
            rocA = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{cls} (AUC={rocA:.3f})", color=colors[i])
        plt.plot([0,1],[0,1], "--", color=plt.cm.Blues(0.35))
        plt.title(title)
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.legend()
        plt.show()

    def plot_confusions_2x2(self, y_true: np.ndarray, y_pred: np.ndarray, title: str):
        if not self.cfg.SHOW_PLOTS: return
        # One 2x2 per class (TP/TN/FP/FN)
        for i, cls in enumerate(self.cfg.CLASS_NAMES):
            yt = y_true[:, i].astype(int)
            yp = y_pred[:, i].astype(int)
            TP = int(((yt==1)&(yp==1)).sum())
            TN = int(((yt==0)&(yp==0)).sum())
            FP = int(((yt==0)&(yp==1)).sum())
            FN = int(((yt==1)&(yp==0)).sum())
            mat = np.array([[TP, FN],
                            [FP, TN]], dtype=int)
            plt.figure(figsize=(3.6,3.2))
            plt.imshow(mat, cmap="Blues")
            plt.title(f"{title} — {cls}")
            plt.xticks([0,1], ["Pred+","Pred-"])
            plt.yticks([0,1], ["True+","True-"])
            for r in range(2):
                for c in range(2):
                    plt.text(c, r, str(mat[r,c]), ha="center", va="center")
            plt.colorbar()
            plt.show()

viz = VisualAgent(cfg)

# =========================
# CELL 6 — Metadata helpers (NO download)
# =========================
def load_ptbxl_metadata(ptbxl_root: str):
    db_path = os.path.join(ptbxl_root, "ptbxl_database.csv")
    scp_path = os.path.join(ptbxl_root, "scp_statements.csv")
    assert os.path.exists(db_path), "Missing ptbxl_database.csv"
    assert os.path.exists(scp_path), "Missing scp_statements.csv"
    db = pd.read_csv(db_path)
    scp = pd.read_csv(scp_path, index_col=0)
    return db, scp

def parse_scp_codes(s):
    try:
        return ast.literal_eval(s) if isinstance(s, str) else {}
    except Exception:
        return {}

# =========================
# CELL 7 — Fast PatientId alignment (fixes your error)
# =========================
def build_y_pid_from_db(cfg: Config, db: pd.DataFrame, scp: pd.DataFrame,
                        order: str = "csv_order", mode: str = "diagnostic_only"):
    """
    Returns y_gen (N,K), pid_gen (N,), file_gen (N,) under a chosen db ordering/mode.

    Key fix for your mismatch:
    - We DO NOT drop all-zero rows.
      (Some diagnostic records may map to classes outside your 4-class set, e.g., HYP.)
    """
    db2 = db.copy()

    if order == "sort_ecg_id" and "ecg_id" in db2.columns:
        db2 = db2.sort_values("ecg_id")
    elif order == "sort_filename_hr" and "filename_hr" in db2.columns:
        db2 = db2.sort_values("filename_hr")
    elif order == "sort_filename_lr" and "filename_lr" in db2.columns:
        db2 = db2.sort_values("filename_lr")
    # else csv_order = keep as-is

    db2["scp_codes_dict"] = db2["scp_codes"].apply(parse_scp_codes)

    diag_codes = set(scp.index[scp.get("diagnostic", 0) == 1].tolist()) if "diagnostic" in scp.columns else set(scp.index.tolist())
    form_codes = set(scp.index[scp.get("form", 0) == 1].tolist()) if "form" in scp.columns else set()
    rhythm_codes = set(scp.index[scp.get("rhythm", 0) == 1].tolist()) if "rhythm" in scp.columns else set()

    allowed_codes = diag_codes
    if mode == "diag_form_rhythm":
        allowed_codes = diag_codes.union(form_codes).union(rhythm_codes)
    elif mode == "all_rows":
        allowed_codes = None  # accept all rows

    super_to_idx = {k: i for i, k in enumerate(cfg.CLASS_NAMES)}

    y_list, pid_list, file_list = [], [], []
    for _, row in db2.iterrows():
        codes = row["scp_codes_dict"].keys()

        if mode != "all_rows":
            has_any = any((c in allowed_codes) for c in codes)
            if not has_any:
                continue

        mh = np.zeros((cfg.N_CLASSES,), dtype=np.int64)

        # Map only DIAGNOSTIC codes to diagnostic_class (NORM/MI/STTC/CD/HYP)
        for c in codes:
            if c in diag_codes:
                dc = scp.loc[c, "diagnostic_class"] if "diagnostic_class" in scp.columns and c in scp.index else None
                if isinstance(dc, str) and dc in super_to_idx:
                    mh[super_to_idx[dc]] = 1

        y_list.append(mh)
        pid_list.append(int(row["patient_id"]))
        file_list.append(str(row["filename_hr"]) if "filename_hr" in row else "")

    y_gen = np.stack(y_list, axis=0) if len(y_list) else np.zeros((0, cfg.N_CLASSES), dtype=np.int64)
    pid_gen = np.array(pid_list, dtype=np.int64)
    file_gen = np.array(file_list, dtype=object)
    return y_gen, pid_gen, file_gen

def ensure_patient_ids_fast(cfg: Config, y_saved: np.ndarray) -> Optional[np.ndarray]:
    pid_path = os.path.join(cfg.RUN_DIR, "patient_ids.npy")
    if os.path.exists(pid_path):
        return np.load(pid_path).astype(np.int64)

    db, scp = load_ptbxl_metadata(cfg.PTBXL_ROOT)
    print("db exists:", True, "| scp exists:", True)
    print("Loaded y_saved:", y_saved.shape)

    y_saved_i = y_saved.astype(np.int64)

    orders = ["csv_order", "sort_ecg_id", "sort_filename_hr", "sort_filename_lr"]
    modes  = ["diagnostic_only", "diag_form_rhythm", "all_rows"]

    found = None
    for od in orders:
        for md in modes:
            print(f"Trying alignment order={od} | mode={md} ...")
            y_gen, pid_gen, file_gen = build_y_pid_from_db(cfg, db, scp, order=od, mode=md)

            if y_gen.shape != y_saved_i.shape:
                continue

            if np.array_equal(y_gen, y_saved_i):
                found = (pid_gen, file_gen, od, md)
                break
        if found is not None:
            break

    if found is None:
        return None

    pid_gen, file_gen, od, md = found
    np.save(pid_path, pid_gen)
    np.save(os.path.join(cfg.RUN_DIR, "record_files.npy"), file_gen)
    with open(os.path.join(cfg.RUN_DIR, "alignment_manifest.json"), "w") as f:
        json.dump({"order": od, "mode": md, "N": int(len(pid_gen))}, f, indent=2)

    print("✅ FAST aligned + saved patient_ids.npy using:", {"order": od, "mode": md, "N": len(pid_gen)})
    return pid_gen

# =========================
# CELL 8 — If fast alignment fails: rebuild aligned dataset from Drive records500
# =========================
import wfdb

def backup_run_dir(cfg: Config):
    ts = time.strftime("%Y%m%d_%H%M%S")
    backup_dir = os.path.join(cfg.RUN_DIR, f"_backup_before_rebuild_{ts}")
    os.makedirs(backup_dir, exist_ok=True)
    for fn in ["X_ptbxl_10s.npy","X_ptbxl_10s_preproc.npy","y_multi_hot.npy","patient_ids.npy","record_files.npy","alignment_manifest.json"]:
        p = os.path.join(cfg.RUN_DIR, fn)
        if os.path.exists(p):
            shutil.copy2(p, os.path.join(backup_dir, fn))
    print("Backed up old arrays to:", backup_dir)

def pad_trunc(sig_ch_l: np.ndarray, L: int):
    # sig: (C, T)
    T = sig_ch_l.shape[1]
    if T == L:
        return sig_ch_l
    if T > L:
        return sig_ch_l[:, :L]
    out = np.zeros((sig_ch_l.shape[0], L), dtype=sig_ch_l.dtype)
    out[:, :T] = sig_ch_l
    return out

def bandpass(cfg: Config, x: np.ndarray) -> np.ndarray:
    nyq = 0.5 * cfg.FS
    low = cfg.BP_LOW / nyq
    high = cfg.BP_HIGH / nyq
    b, a = butter(cfg.BP_ORDER, [low, high], btype="band")
    return filtfilt(b, a, x, axis=-1)

def notch(cfg: Config, x: np.ndarray) -> np.ndarray:
    nyq = 0.5 * cfg.FS
    w0 = cfg.NOTCH_HZ / nyq
    b, a = iirnotch(w0, cfg.NOTCH_Q)
    return filtfilt(b, a, x, axis=-1)

def zscore(x: np.ndarray) -> np.ndarray:
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True) + 1e-8
    return (x - mu) / sd

def rebuild_from_raw_drive(cfg: Config):
    backup_run_dir(cfg)

    db, scp = load_ptbxl_metadata(cfg.PTBXL_ROOT)
    db["scp_codes_dict"] = db["scp_codes"].apply(parse_scp_codes)

    diag_codes = set(scp.index[scp.get("diagnostic", 0) == 1].tolist()) if "diagnostic" in scp.columns else set(scp.index.tolist())
    super_to_idx = {k: i for i, k in enumerate(cfg.CLASS_NAMES)}

    # Select diagnostic rows (this is typically N=21388)
    keep_rows = []
    for idx, row in db.iterrows():
        codes = row["scp_codes_dict"].keys()
        if any((c in diag_codes) for c in codes):
            keep_rows.append(idx)

    dbk = db.loc[keep_rows].reset_index(drop=True)
    N = len(dbk)
    print("Rebuilding diagnostic subset N =", N)

    # Write X directly to .npy memmap (no huge RAM spikes)
    X_path = os.path.join(cfg.RUN_DIR, "X_ptbxl_10s.npy")
    Xmm = np.lib.format.open_memmap(X_path, mode="w+", dtype="float32", shape=(N, cfg.N_LEADS, cfg.SIG_LEN))

    y = np.zeros((N, cfg.N_CLASSES), dtype=np.int64)
    pid = np.zeros((N,), dtype=np.int64)
    files = np.empty((N,), dtype=object)

    for i in tqdm(range(N), desc="Reading records500 from Drive"):
        row = dbk.iloc[i]
        rec_rel = row["filename_hr"]  # e.g. records500/xxxxx/xxxxx_hr
        rec_path = os.path.join(cfg.PTBXL_ROOT, rec_rel)

        sig, meta = wfdb.rdsamp(rec_path)  # (T, C)
        sig = sig.T.astype(np.float32)     # (C, T)
        sig = pad_trunc(sig, cfg.SIG_LEN)

        # Optional preprocess on-the-fly
        if cfg.PRE_BANDPASS:
            sig = bandpass(cfg, sig)
        if cfg.PRE_NOTCH:
            sig = notch(cfg, sig)
        if cfg.PRE_ZSCORE:
            sig = zscore(sig)

        Xmm[i] = sig.astype(np.float32)

        mh = np.zeros((cfg.N_CLASSES,), dtype=np.int64)
        for c in row["scp_codes_dict"].keys():
            if c in diag_codes:
                dc = scp.loc[c, "diagnostic_class"] if "diagnostic_class" in scp.columns and c in scp.index else None
                if isinstance(dc, str) and dc in super_to_idx:
                    mh[super_to_idx[dc]] = 1
        y[i] = mh
        pid[i] = int(row["patient_id"])
        files[i] = str(rec_rel)

    Xmm.flush()
    np.save(os.path.join(cfg.RUN_DIR, "y_multi_hot.npy"), y)
    np.save(os.path.join(cfg.RUN_DIR, "patient_ids.npy"), pid)
    np.save(os.path.join(cfg.RUN_DIR, "record_files.npy"), files)

    with open(os.path.join(cfg.RUN_DIR, "alignment_manifest.json"), "w") as f:
        json.dump({"rebuilt_from_raw": True, "N": int(N), "ptbxl_root": cfg.PTBXL_ROOT}, f, indent=2)

    print("✅ Rebuild done. Saved X/y/patient_ids/record_files to RUN_DIR.")
    return np.asarray(Xmm), y, pid

# =========================
# CELL 9 — Load data (prefer existing X, fix patient_ids fast, else rebuild)
# =========================
def load_main_arrays(cfg: Config):
    os.makedirs(cfg.RUN_DIR, exist_ok=True)

    y_path = os.path.join(cfg.RUN_DIR, "y_multi_hot.npy")
    assert os.path.exists(y_path), "Missing y_multi_hot.npy in RUN_DIR"
    y = np.load(y_path).astype(np.int64)

    # Choose X
    X = None
    x_used = None
    for fn in ["X_ptbxl_10s_preproc.npy", "X_ptbxl_10s.npy"]:
        p = os.path.join(cfg.RUN_DIR, fn)
        if os.path.exists(p):
            X = np.load(p, mmap_mode="r")  # memory-friendly
            x_used = fn
            break

    if X is None:
        print("No X arrays in RUN_DIR -> will rebuild from Drive records500.")
        pid = None
    else:
        pid = ensure_patient_ids_fast(cfg, y_saved=y)

    if pid is None:
        print("patient_ids.npy missing and cannot be safely inferred -> rebuilding aligned dataset from Drive records500...")
        X, y, pid = rebuild_from_raw_drive(cfg)
        x_used = "rebuilt_X_ptbxl_10s.npy"

    print("Loaded:", {"X": str(getattr(X, "shape", None)), "y": y.shape, "pid": pid.shape, "x_used": x_used})
    return X, y, pid, x_used

X, y, patient_ids, x_used = load_main_arrays(cfg)

# =========================
# CELL 10 — Dataset / Models
# =========================
class ECGDataset(Dataset):
    def __init__(self, X, y, idx, cfg: Config, augment=False):
        self.X = X
        self.y = y
        self.idx = np.array(idx, dtype=np.int64)
        self.cfg = cfg
        self.augment = augment

    def __len__(self):
        return len(self.idx)

    def _augment(self, x):
        scale = np.random.uniform(0.90, 1.10)
        x = x * scale
        if x.shape[-1] > 10:
            shift = np.random.randint(-50, 51)
            x = np.roll(x, shift, axis=-1)
        x = x + np.random.normal(0, 0.01, size=x.shape).astype(np.float32)
        return x.astype(np.float32)

    def __getitem__(self, i):
        j = self.idx[i]
        x = np.array(self.X[j], dtype=np.float32)  # ensure concrete (handles memmap)
        if self.augment:
            x = self._augment(x)
        yy = self.y[j].astype(np.float32)
        return torch.from_numpy(x).float(), torch.from_numpy(yy).float()

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=7, s=1, p=None, dropout=0.0):
        super().__init__()
        if p is None: p = k // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.act(self.bn(self.conv(x))))

class ResBlock1D(nn.Module):
    def __init__(self, ch, dropout=0.0):
        super().__init__()
        self.c1 = ConvBNAct(ch, ch, k=7, s=1, dropout=dropout)
        self.c2 = ConvBNAct(ch, ch, k=7, s=1, dropout=0.0)
    def forward(self, x):
        return x + self.c2(self.c1(x))

class CNNBaseline(nn.Module):
    def __init__(self, cfg: Config, in_ch: int):
        super().__init__()
        chs = cfg.CNN_CHANNELS
        self.stem = nn.Sequential(
            ConvBNAct(in_ch, chs[0], k=11, s=2, dropout=cfg.DROPOUT),
            ResBlock1D(chs[0], dropout=cfg.DROPOUT),
        )
        self.stage2 = nn.Sequential(
            ConvBNAct(chs[0], chs[1], k=9, s=2, dropout=cfg.DROPOUT),
            ResBlock1D(chs[1], dropout=cfg.DROPOUT),
        )
        self.stage3 = nn.Sequential(
            ConvBNAct(chs[1], chs[2], k=7, s=2, dropout=cfg.DROPOUT),
            ResBlock1D(chs[2], dropout=cfg.DROPOUT),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(chs[2], cfg.N_CLASSES)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x)
        return self.head(x)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 8192):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T, :]

class CNNTransformerHybrid(nn.Module):
    def __init__(self, cfg: Config, in_ch: int):
        super().__init__()
        self.cnn = nn.Sequential(
            ConvBNAct(in_ch, cfg.EMBED_DIM, k=11, s=2, dropout=cfg.DROPOUT),
            ResBlock1D(cfg.EMBED_DIM, dropout=cfg.DROPOUT),
            ConvBNAct(cfg.EMBED_DIM, cfg.EMBED_DIM, k=9, s=2, dropout=cfg.DROPOUT),
            ResBlock1D(cfg.EMBED_DIM, dropout=cfg.DROPOUT),
            ConvBNAct(cfg.EMBED_DIM, cfg.EMBED_DIM, k=7, s=2, dropout=cfg.DROPOUT),
        )
        self.pos = PositionalEncoding(cfg.EMBED_DIM, max_len=8192)
        enc = nn.TransformerEncoderLayer(
            d_model=cfg.EMBED_DIM,
            nhead=cfg.N_HEADS,
            dim_feedforward=cfg.FF_DIM,
            dropout=cfg.DROPOUT,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=cfg.N_LAYERS)
        self.norm = nn.LayerNorm(cfg.EMBED_DIM)
        self.head = nn.Sequential(nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.EMBED_DIM, cfg.N_CLASSES))
    def forward(self, x):
        x = self.cnn(x)          # (B,D,T)
        x = x.transpose(1, 2)    # (B,T,D)
        x = self.pos(x)
        x = self.tr(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)

def build_model(cfg: Config, model_type: str, in_ch: int):
    if model_type == "cnn":
        return CNNBaseline(cfg, in_ch)
    if model_type == "hybrid":
        return CNNTransformerHybrid(cfg, in_ch)
    raise ValueError("model_type must be cnn|hybrid")

# =========================
# CELL 11 — Metrics (all required)
# =========================
def multilabel_confusion_counts(y_true, y_pred):
    TP = (y_true * y_pred).sum(axis=0)
    TN = ((1 - y_true) * (1 - y_pred)).sum(axis=0)
    FP = ((1 - y_true) * y_pred).sum(axis=0)
    FN = (y_true * (1 - y_pred)).sum(axis=0)
    return TP, TN, FP, FN

def metrics_from_probs(cfg: Config, y_true: np.ndarray, y_prob: np.ndarray):
    y_pred = (y_prob >= cfg.THR).astype(np.int64)
    TP, TN, FP, FN = multilabel_confusion_counts(y_true, y_pred)

    per_acc  = safe_div(TP + TN, TP + TN + FP + FN)
    per_prec = safe_div(TP, TP + FP)
    per_rec  = safe_div(TP, TP + FN)   # recall/sensitivity
    per_spec = safe_div(TN, TN + FP)
    per_f1   = safe_div(2 * per_prec * per_rec, per_prec + per_rec)

    per_auc = []
    for k in range(cfg.N_CLASSES):
        yt = y_true[:, k]
        yp = y_prob[:, k]
        if len(np.unique(yt)) < 2:
            per_auc.append(np.nan)
        else:
            per_auc.append(float(roc_auc_score(yt, yp)))
    macro_auc = float(np.nanmean(per_auc))

    macro = dict(
        ACC=float(per_acc.mean()),
        P=float(per_prec.mean()),
        R=float(per_rec.mean()),
        SENS=float(per_rec.mean()),
        SPEC=float(per_spec.mean()),
        F1=float(per_f1.mean()),
        AUROC=macro_auc,
    )

    TPm, TNm, FPm, FNm = TP.sum(), TN.sum(), FP.sum(), FN.sum()
    micro_prec = float(safe_div(TPm, TPm + FPm))
    micro_rec  = float(safe_div(TPm, TPm + FNm))
    micro_f1   = float(safe_div(2 * micro_prec * micro_rec, micro_prec + micro_rec))
    subset_acc = float((y_pred == y_true).all(axis=1).mean())

    per_class = {}
    for i, name in enumerate(cfg.CLASS_NAMES):
        per_class[name] = {
            "ACC": float(per_acc[i]),
            "P": float(per_prec[i]),
            "R": float(per_rec[i]),
            "SENS": float(per_rec[i]),
            "SPEC": float(per_spec[i]),
            "F1": float(per_f1[i]),
            "AUROC": None if np.isnan(per_auc[i]) else float(per_auc[i]),
        }

    return {
        "thr": cfg.THR,
        "macro": macro,
        "micro": {"P": micro_prec, "R": micro_rec, "F1": micro_f1},
        "subset_acc": subset_acc,
        "per_class": per_class,
        "y_pred": y_pred,
    }

def bootstrap_ci_patientwise(cfg: Config, y_true, y_prob, patient_ids):
    rng = np.random.default_rng(cfg.SEED)
    uniq_p = np.unique(patient_ids)
    keys = ["ACC","P","R","SENS","SPEC","F1","AUROC"]
    stats = {k: [] for k in keys}

    for _ in range(cfg.CI_N_BOOT):
        samp_p = rng.choice(uniq_p, size=len(uniq_p), replace=True)
        m = np.isin(patient_ids, samp_p)
        if m.sum() == 0:
            continue
        mm = metrics_from_probs(cfg, y_true[m], y_prob[m])["macro"]
        for k in keys:
            stats[k].append(mm[k])

    def ci(arr):
        arr = np.array(arr, dtype=np.float64)
        arr = arr[np.isfinite(arr)]
        lo = float(np.quantile(arr, cfg.CI_ALPHA/2))
        hi = float(np.quantile(arr, 1 - cfg.CI_ALPHA/2))
        return lo, hi

    out = {}
    for k in keys:
        out[k] = {"mean": float(np.mean(stats[k])), "ci95": ci(stats[k])}
    return out

def summarize_metrics(metrics: Dict, title: str, ci: Optional[Dict] = None):
    m = metrics["macro"]
    s = (f"{title}\n"
         f"Macro ACC={m['ACC']:.4f} | P={m['P']:.4f} | R/Sens={m['R']:.4f} | "
         f"Spec={m['SPEC']:.4f} | F1={m['F1']:.4f} | AUROC={m['AUROC']:.4f}\n"
         f"Micro P={metrics['micro']['P']:.4f} | Micro R={metrics['micro']['R']:.4f} | Micro F1={metrics['micro']['F1']:.4f} | SubsetAcc={metrics['subset_acc']:.4f}\n"
         f"Per-class: " + ", ".join([f"{k}:F1={v['F1']:.3f},AUC={(v['AUROC'] if v['AUROC'] is not None else np.nan):.3f}" for k,v in metrics["per_class"].items()])
    )
    if ci is not None:
        s += ("\n95% CI (patient-wise bootstrap): "
              f"F1={ci['F1']['mean']:.4f}[{ci['F1']['ci95'][0]:.4f},{ci['F1']['ci95'][1]:.4f}] | "
              f"AUROC={ci['AUROC']['mean']:.4f}[{ci['AUROC']['ci95'][0]:.4f},{ci['AUROC']['ci95'][1]:.4f}] | "
              f"SENS={ci['SENS']['mean']:.4f}[{ci['SENS']['ci95'][0]:.4f},{ci['SENS']['ci95'][1]:.4f}] | "
              f"SPEC={ci['SPEC']['mean']:.4f}[{ci['SPEC']['ci95'][0]:.4f},{ci['SPEC']['ci95'][1]:.4f}]")
    print(s)
    print("Note: Research prototype output — not for clinical use.\n")

# =========================
# CELL 12 — SplitAgent (patient-wise)
# =========================
class SplitAgent:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _patient_level_labels(self, y, patient_ids):
        uniq = np.unique(patient_ids)
        idx = {p:i for i,p in enumerate(uniq)}
        yp = np.zeros((len(uniq), y.shape[1]), dtype=np.int64)
        for i in range(len(patient_ids)):
            yp[idx[patient_ids[i]]] = np.maximum(yp[idx[patient_ids[i]]], y[i])
        return uniq, yp

    def _combo_labels(self, yp):
        bits = (1 << np.arange(yp.shape[1], dtype=np.int64))
        return (yp.astype(np.int64) * bits[None,:]).sum(axis=1).astype(np.int64)

    def _score_split(self, combo_all, combo_train, combo_val, combo_test):
        def dist(c):
            vals, cnt = np.unique(c, return_counts=True)
            p = np.zeros((max(int(combo_all.max()), 1)+1,), dtype=np.float64)
            p[vals] = cnt / max(cnt.sum(), 1)
            return p
        p_all = dist(combo_all)
        return float(np.abs(dist(combo_train)-p_all).mean() + np.abs(dist(combo_val)-p_all).mean() + np.abs(dist(combo_test)-p_all).mean())

    def fixed_split_70_15_15(self, y, patient_ids):
        cfg = self.cfg
        patients, yp = self._patient_level_labels(y, patient_ids)
        combo = self._combo_labels(yp)

        P = len(patients)
        n_test = int(round(P * cfg.TEST_FRAC))
        n_val  = int(round(P * cfg.VAL_FRAC))
        n_train = P - n_test - n_val
        assert n_train > 0 and n_val > 0 and n_test > 0

        best = None
        rng = np.random.default_rng(cfg.SEED)
        for _ in range(cfg.SPLIT_SEARCH_TRIALS):
            perm = rng.permutation(P)
            te = perm[:n_test]
            va = perm[n_test:n_test+n_val]
            tr = perm[n_test+n_val:]
            sc = self._score_split(combo, combo[tr], combo[va], combo[te])
            if best is None or sc < best[0]:
                best = (sc, tr, va, te)

        _, tr, va, te = best
        p_tr, p_va, p_te = patients[tr], patients[va], patients[te]
        tr_idx = np.where(np.isin(patient_ids, p_tr))[0]
        va_idx = np.where(np.isin(patient_ids, p_va))[0]
        te_idx = np.where(np.isin(patient_ids, p_te))[0]
        return tr_idx, va_idx, te_idx

    def cv_folds_on_trainval(self, y, patient_ids, trainval_patients):
        cfg = self.cfg
        mask_tv = np.isin(patient_ids, trainval_patients)
        y_tv = y[mask_tv]
        pid_tv = patient_ids[mask_tv]

        p_tv, yp_tv = self._patient_level_labels(y_tv, pid_tv)
        combo_tv = self._combo_labels(yp_tv)

        folds_of_patient = {int(p): -1 for p in p_tv}
        fold_counts = [dict() for _ in range(cfg.FOLDS)]

        vals, cnt = np.unique(combo_tv, return_counts=True)
        freq = dict(zip(vals.tolist(), cnt.tolist()))
        rarity = np.array([1.0/(freq[int(c)]+1e-6) for c in combo_tv])
        order = np.argsort(-rarity)

        for pi in order:
            p = int(p_tv[pi]); c = int(combo_tv[pi])
            best_f, best_score = None, None
            for f in range(cfg.FOLDS):
                cur = fold_counts[f].get(c, 0)
                tot = sum(fold_counts[f].values()) + 1e-9
                score = (cur / tot) + (tot * 1e-6)
                if best_score is None or score < best_score:
                    best_score = score; best_f = f
            folds_of_patient[p] = best_f
            fold_counts[best_f][c] = fold_counts[best_f].get(c, 0) + 1

        fold_ids = np.full((len(patient_ids),), -1, dtype=np.int64)
        for i in range(len(patient_ids)):
            if mask_tv[i]:
                fold_ids[i] = folds_of_patient[int(patient_ids[i])]
        return fold_ids

# =========================
# CELL 13 — Train/Eval
# =========================
class TrainAgent:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _pos_weight(self, y_train):
        if not self.cfg.USE_POS_WEIGHT:
            return None
        pos = y_train.sum(axis=0).astype(np.float64)
        neg = y_train.shape[0] - pos
        pw = (neg + 1e-6) / (pos + 1e-6)
        return torch.tensor(pw, dtype=torch.float32, device=self.cfg.DEVICE)

    def criterion(self, y_train):
        return nn.BCEWithLogitsLoss(pos_weight=self._pos_weight(y_train))

    def fit(self, model, train_loader, val_loader, y_train_np, save_path, tag):
        cfg = self.cfg
        model = model.to(cfg.DEVICE)
        crit = self.criterion(y_train_np)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=cfg.PLATEAU_FACTOR, patience=cfg.PLATEAU_PATIENCE)
        scaler = torch.cuda.amp.GradScaler(enabled=(cfg.DEVICE=="cuda"))

        best_val = 1e18
        best_state = None
        no_imp = 0
        hist = {"train_loss": [], "val_loss": []}

        for ep in range(cfg.MAX_EPOCHS):
            model.train()
            opt.zero_grad(set_to_none=True)
            tr_loss = 0.0

            for step,(xb,yb) in enumerate(train_loader):
                xb = xb.to(cfg.DEVICE, non_blocking=True)
                yb = yb.to(cfg.DEVICE, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=(cfg.DEVICE=="cuda")):
                    logits = model(xb)
                    loss = crit(logits, yb) / cfg.GRAD_ACCUM_STEPS

                scaler.scale(loss).backward()

                if cfg.CLIP_GRAD_NORM is not None:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD_NORM)

                if (step+1) % cfg.GRAD_ACCUM_STEPS == 0:
                    scaler.step(opt); scaler.update()
                    opt.zero_grad(set_to_none=True)

                tr_loss += loss.item() * cfg.GRAD_ACCUM_STEPS

            tr_loss /= max(len(train_loader), 1)

            model.eval()
            va_loss = 0.0
            with torch.no_grad():
                for xb,yb in val_loader:
                    xb = xb.to(cfg.DEVICE)
                    yb = yb.to(cfg.DEVICE)
                    logits = model(xb)
                    va_loss += crit(logits, yb).item()
            va_loss /= max(len(val_loader), 1)

            sch.step(va_loss)
            hist["train_loss"].append(tr_loss)
            hist["val_loss"].append(va_loss)

            improved = va_loss < best_val - 1e-6
            if improved:
                best_val = va_loss
                best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                no_imp = 0
                if cfg.SAVE_BEST and save_path:
                    torch.save(best_state, save_path)
            else:
                no_imp += 1

            print(f"{tag} | epoch {ep:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | best={best_val:.4f}")

            if no_imp >= cfg.EARLY_STOP_PATIENCE:
                print(f"{tag} early stopping at epoch {ep:02d}")
                break

        if best_state is None:
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        return best_state, hist

class EvalAgent:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def predict_proba(self, model, loader):
        cfg = self.cfg
        model.eval()
        logits_list, y_list = [], []
        with torch.no_grad():
            for xb,yb in loader:
                xb = xb.to(cfg.DEVICE)
                logits = model(xb).detach().cpu().numpy()
                logits_list.append(logits)
                y_list.append(yb.numpy())
        logits = np.concatenate(logits_list, axis=0)
        y_true = np.concatenate(y_list, axis=0).astype(np.int64)
        y_prob = sigmoid_np(logits)
        return y_true, y_prob

# =========================
# CELL 14 — Orchestrator (fixed split + CV + plots)
# =========================
def make_loaders(cfg, X, y, tr_idx, va_idx):
    train_ds = ECGDataset(X, y, tr_idx, cfg, augment=True)
    val_ds   = ECGDataset(X, y, va_idx, cfg, augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader

def run_model(cfg: Config, model_type: str, X, y, patient_ids):
    seed_everything(cfg.SEED)
    split = SplitAgent(cfg)
    trainer = TrainAgent(cfg)
    evaluator = EvalAgent(cfg)

    tr_idx, va_idx, te_idx = split.fixed_split_70_15_15(y, patient_ids)
    print("Fixed split sizes:", {"train":len(tr_idx),"val":len(va_idx),"test":len(te_idx)},
          "| patients:", {"train":len(np.unique(patient_ids[tr_idx])),
                         "val":len(np.unique(patient_ids[va_idx])),
                         "test":len(np.unique(patient_ids[te_idx]))})

    in_ch = X.shape[1]

    # --- Fixed training
    train_loader, val_loader = make_loaders(cfg, X, y, tr_idx, va_idx)
    test_loader = DataLoader(ECGDataset(X, y, te_idx, cfg, augment=False), batch_size=cfg.BATCH_SIZE, shuffle=False,
                             num_workers=cfg.NUM_WORKERS, pin_memory=True)

    model = build_model(cfg, model_type, in_ch).to(cfg.DEVICE)
    save_path = os.path.join(cfg.RUN_DIR, f"best_{model_type}_fixed.pt")
    best_state, history = trainer.fit(model, train_loader, val_loader, y[tr_idx], save_path, tag=f"{model_type.upper()} FIXED")

    model.load_state_dict(best_state, strict=True)
    model.to(cfg.DEVICE)

    # VAL
    yv_true, yv_prob = evaluator.predict_proba(model, val_loader)
    val_m = metrics_from_probs(cfg, yv_true, yv_prob)
    val_ci = bootstrap_ci_patientwise(cfg, yv_true, yv_prob, patient_ids[va_idx]) if cfg.CI_PATIENTWISE else None

    # TEST
    yt_true, yt_prob = evaluator.predict_proba(model, test_loader)
    test_m = metrics_from_probs(cfg, yt_true, yt_prob)
    test_ci = bootstrap_ci_patientwise(cfg, yt_true, yt_prob, patient_ids[te_idx]) if cfg.CI_PATIENTWISE else None

    summarize_metrics(val_m, f"[{model_type}] FIXED VAL", ci=val_ci)
    summarize_metrics(test_m, f"[{model_type}] FIXED TEST", ci=test_ci)

    # --- Visuals (blue hues)
    viz.plot_history(history, f"{model_type.upper()} — training curves (fixed split)")
    viz.plot_macro_bar([val_m["macro"], test_m["macro"]], ["VAL","TEST"], f"{model_type.upper()} — Macro metrics")
    viz.plot_roc(yv_true, yv_prob, f"{model_type.upper()} — ROC (VAL)")
    viz.plot_roc(yt_true, yt_prob, f"{model_type.upper()} — ROC (TEST)")
    viz.plot_confusions_2x2(yv_true, val_m["y_pred"], f"{model_type.upper()} — Confusions (VAL)")
    viz.plot_confusions_2x2(yt_true, test_m["y_pred"], f"{model_type.upper()} — Confusions (TEST)")

    # --- 5-fold CV on train+val (unseen test)
    trainval_patients = np.unique(patient_ids[np.concatenate([tr_idx, va_idx])])
    fold_ids = split.cv_folds_on_trainval(y, patient_ids, trainval_patients)

    tv_idx = np.where(fold_ids >= 0)[0]
    fold_f1, fold_auc = [], []
    fold_histories = []

    oof_true_list, oof_prob_list, oof_pid_list = [], [], []

    for f in range(cfg.FOLDS):
        va_f = np.where(fold_ids == f)[0]
        tr_f = np.setdiff1d(tv_idx, va_f)

        tr_loader, va_loader = make_loaders(cfg, X, y, tr_f, va_f)
        model = build_model(cfg, model_type, in_ch).to(cfg.DEVICE)
        save_path = os.path.join(cfg.RUN_DIR, f"best_{model_type}_fold{f}.pt")

        best_state, hist = trainer.fit(model, tr_loader, va_loader, y[tr_f], save_path, tag=f"{model_type.upper()} FOLD{f}")
        fold_histories.append(hist)

        model.load_state_dict(best_state, strict=True)
        model.to(cfg.DEVICE)
        yt, yp = evaluator.predict_proba(model, va_loader)
        m = metrics_from_probs(cfg, yt, yp)

        fold_f1.append(m["macro"]["F1"])
        fold_auc.append(m["macro"]["AUROC"])

        oof_true_list.append(yt)
        oof_prob_list.append(yp)
        oof_pid_list.append(patient_ids[va_f])

        print(f"{model_type.upper()} fold {f} macro:", m["macro"])

    oof_true = np.concatenate(oof_true_list, axis=0)
    oof_prob = np.concatenate(oof_prob_list, axis=0)
    oof_pid  = np.concatenate(oof_pid_list,  axis=0)

    oof_m = metrics_from_probs(cfg, oof_true, oof_prob)
    oof_ci = bootstrap_ci_patientwise(cfg, oof_true, oof_prob, oof_pid) if cfg.CI_PATIENTWISE else None
    summarize_metrics(oof_m, f"[{model_type}] CV OOF (train+val)", ci=oof_ci)

    # CV visuals
    if cfg.SHOW_PLOTS:
        colors = plt.cm.Blues(np.linspace(0.55, 0.9, cfg.FOLDS))
        plt.figure(figsize=(7,3))
        plt.bar(np.arange(cfg.FOLDS), fold_f1, color=colors)
        plt.title(f"{model_type.upper()} — Fold macro-F1")
        plt.ylim(0,1)
        plt.show()

        plt.figure(figsize=(7,3))
        plt.bar(np.arange(cfg.FOLDS), fold_auc, color=colors)
        plt.title(f"{model_type.upper()} — Fold macro-AUROC")
        plt.ylim(0,1)
        plt.show()

    return {
        "fixed_val": val_m, "fixed_test": test_m,
        "cv_oof": oof_m,
        "fold_f1": fold_f1, "fold_auc": fold_auc
    }

# =========================
# CELL 15 — RUN ALL MODELS + Paired t-test
# =========================
all_results = {}
for mt in cfg.RUN_MODELS:
    print("\n" + "="*110)
    print("RUNNING:", mt)
    print("="*110)
    all_results[mt] = run_model(cfg, mt, X, y, patient_ids)

# Paired t-test (fold macro-F1) CNN vs HYBRID
if "cnn" in all_results and "hybrid" in all_results:
    a = all_results["cnn"]["fold_f1"]
    b = all_results["hybrid"]["fold_f1"]
    t, p = ttest_rel(a, b)
    print(f"\nPaired t-test (fold macro-F1) cnn vs hybrid: t={t:.4f}, p={p:.6f}")

    a2 = all_results["cnn"]["fold_auc"]
    b2 = all_results["hybrid"]["fold_auc"]
    t2, p2 = ttest_rel(a2, b2)
    print(f"Paired t-test (fold macro-AUROC) cnn vs hybrid: t={t2:.4f}, p={p2:.6f}")

# Save summary json
out_path = os.path.join(cfg.RUN_DIR, "MAIN_PIPELINE_results_summary.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved:", out_path)

In [ ]:
# ============================================================
# EVAL-ONLY: Precision/Recall/F1/Accuracy + ROC-AUC/AUPRC
# + Bootstrap 95% CI
# + Per-fold metrics (for paired t-test)
# ============================================================

import os, glob, json
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    multilabel_confusion_matrix
)

# -----------------------
# CONFIG
# -----------------------
RUN_A = "${DRIVE_ROOT}/cardioai_logs/ptbxl_rerun_20260204_004052"   # Hybrid
RUN_B = None  # <-- set to baseline RUN DIR later to run paired t-test (must have oof_prob + fold_ids)
SOURCE_DIR = "${DRIVE_ROOT}/cardioai_logs/ptbxl_build_20260128_173743"

THR = 0.5
N_BOOT = 1000
SEED = 7

OUT_DIR = os.path.join(RUN_A, "report_stats")
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# HELPERS
# -----------------------
def find_first(base, patterns):
    cands = []
    for pat in patterns:
        cands += glob.glob(os.path.join(base, pat), recursive=True)
    cands = sorted(list(set(cands)))
    return cands[0] if cands else None

def load_y_multi_hot(run_dir, source_dir):
    cands = [
        os.path.join(run_dir, "artifacts", "y_multi_hot.npy"),
        os.path.join(run_dir, "y_multi_hot.npy"),
        os.path.join(source_dir, "y_multi_hot.npy"),
        os.path.join(source_dir, "artifacts", "y_multi_hot.npy"),
    ]
    for p in cands:
        if os.path.exists(p):
            return np.load(p).astype(int), p
    raise FileNotFoundError("Could not find y_multi_hot.npy in RUN or SOURCE.")

def extract_test_idx(split_json):
    for k in ["test_idx", "test", "idx_test"]:
        if k in split_json and isinstance(split_json[k], list):
            return np.array(split_json[k], dtype=int)
        if k in split_json and isinstance(split_json[k], dict) and "idx" in split_json[k]:
            return np.array(split_json[k]["idx"], dtype=int)
    if "test" in split_json and isinstance(split_json["test"], dict) and "idx" in split_json["test"]:
        return np.array(split_json["test"]["idx"], dtype=int)
    raise KeyError(f"No test indices found. Keys: {list(split_json.keys())}")

def hamming_accuracy(y_true, y_pred):
    return float(1.0 - np.mean(y_true != y_pred))

def label_macro_accuracy(y_true, y_pred):
    mcm = multilabel_confusion_matrix(y_true, y_pred)  # (L,2,2)
    tn = mcm[:,0,0]; fp = mcm[:,0,1]; fn = mcm[:,1,0]; tp = mcm[:,1,1]
    acc = (tp + tn) / np.maximum(tp + tn + fp + fn, 1)
    return float(np.mean(acc))

def specificity_macro_micro(y_true, y_pred):
    mcm = multilabel_confusion_matrix(y_true, y_pred)
    tn = mcm[:,0,0]; fp = mcm[:,0,1]
    spec_per_label = tn / np.maximum(tn + fp, 1)
    spec_macro = float(np.mean(spec_per_label))
    spec_micro = float(np.sum(tn) / max(np.sum(tn) + np.sum(fp), 1.0))
    return spec_macro, spec_micro

def safe_auc_macro(y_true, y_prob):
    vals = []
    for k in range(y_true.shape[1]):
        yt = y_true[:, k]
        if yt.min() == yt.max():
            continue
        try:
            vals.append(roc_auc_score(yt, y_prob[:, k]))
        except Exception:
            pass
    return float(np.mean(vals)) if len(vals) else None

def safe_auprc_macro(y_true, y_prob):
    vals = []
    for k in range(y_true.shape[1]):
        yt = y_true[:, k]
        if yt.min() == yt.max():
            continue
        try:
            vals.append(average_precision_score(yt, y_prob[:, k]))
        except Exception:
            pass
    return float(np.mean(vals)) if len(vals) else None

def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)

    out = {}
    # "Accuracy" for multilabel (report both)
    out["subset_accuracy_exact_match"] = float(accuracy_score(y_true, y_pred))  # strict
    out["hamming_accuracy"]            = hamming_accuracy(y_true, y_pred)       # label-wise
    out["label_macro_accuracy"]        = label_macro_accuracy(y_true, y_pred)   # avg per label

    out["precision_micro"] = float(precision_score(y_true, y_pred, average="micro", zero_division=0))
    out["recall_micro"]    = float(recall_score(y_true, y_pred, average="micro", zero_division=0))
    out["f1_micro"]        = float(f1_score(y_true, y_pred, average="micro", zero_division=0))

    out["precision_macro"] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    out["recall_macro"]    = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    out["f1_macro"]        = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

    spec_macro, spec_micro = specificity_macro_micro(y_true, y_pred)
    out["specificity_macro"] = spec_macro
    out["specificity_micro"] = spec_micro

    out["roc_auc_macro"] = safe_auc_macro(y_true, y_prob)
    out["auprc_macro"]   = safe_auprc_macro(y_true, y_prob)
    return out

def bootstrap_ci(y_true, y_prob, thr=0.5, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    point = compute_metrics(y_true, y_prob, thr=thr)
    keys = list(point.keys())
    samples = {k: [] for k in keys}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        m = compute_metrics(y_true[idx], y_prob[idx], thr=thr)
        for k in keys:
            if m[k] is None:
                continue
            samples[k].append(m[k])

    out = {}
    for k in keys:
        if len(samples[k]) == 0:
            out[k] = {"point": point[k], "ci95": None}
        else:
            arr = np.array(samples[k], dtype=float)
            lo, hi = np.percentile(arr, [2.5, 97.5])
            out[k] = {"point": float(point[k]) if point[k] is not None else None,
                      "ci95": [float(lo), float(hi)]}
    return out

def load_oof_and_fold_ids(run_dir):
    oof_path = find_first(run_dir, ["**/oof_prob.npy", "**/oof_prob_partial.npy", "**/*oof*prob*.npy"])
    if oof_path is None:
        raise FileNotFoundError("No oof_prob*.npy found in RUN_DIR")

    fold_ids_path = find_first(run_dir, ["**/fold_ids_5fold.npy", "**/*fold_ids*5fold*.npy"])
    if fold_ids_path is None:
        raise FileNotFoundError("No fold_ids_5fold.npy found in RUN_DIR")

    y_prob_oof = np.load(oof_path)
    fold_ids = np.load(fold_ids_path)

    mask = np.isfinite(y_prob_oof).all(axis=1) & (np.abs(y_prob_oof).sum(axis=1) > 0)
    return y_prob_oof, fold_ids, mask, oof_path, fold_ids_path

def fold_metrics_from_oof(y_true_all, y_prob_oof, fold_ids, mask, thr=0.5):
    folds = []
    for f in sorted([int(x) for x in np.unique(fold_ids) if int(x) >= 0]):
        idx = np.where((fold_ids == f) & mask)[0]
        if len(idx) == 0:
            folds.append({"fold": f, "n": 0})
        else:
            m = compute_metrics(y_true_all[idx], y_prob_oof[idx], thr=thr)
            m["fold"] = f
            m["n"] = int(len(idx))
            folds.append(m)
    return folds

def paired_ttest_from_folds(folds_A, folds_B, metric_key="f1_macro"):
    try:
        from scipy.stats import ttest_rel
    except Exception:
        !pip -q install scipy
        from scipy.stats import ttest_rel

    a_vals, b_vals = [], []
    # match folds by fold id
    b_by_fold = {d.get("fold"): d for d in folds_B if "fold" in d}
    for fa in folds_A:
        f = fa.get("fold")
        fb = b_by_fold.get(f)
        if fb is None:
            continue
        if fa.get(metric_key) is None or fb.get(metric_key) is None:
            continue
        a_vals.append(fa[metric_key])
        b_vals.append(fb[metric_key])

    a_vals = np.array(a_vals, dtype=float)
    b_vals = np.array(b_vals, dtype=float)

    if len(a_vals) < 2:
        return {"metric": metric_key, "n_pairs": int(len(a_vals)), "t": None, "p": None}

    t, p = ttest_rel(a_vals, b_vals)
    return {"metric": metric_key, "n_pairs": int(len(a_vals)), "t": float(t), "p": float(p)}

# -----------------------
# LOAD y_true
# -----------------------
y_true_all, y_path = load_y_multi_hot(RUN_A, SOURCE_DIR)
print("✅ y_true:", y_true_all.shape, "from", y_path)

# -----------------------
# HOLDOUT (optional if prob file exists)
# -----------------------
split_path = os.path.join(RUN_A, "split_70_15_15.json")
holdout_prob_path = find_first(RUN_A, [
    "**/holdout_y_prob.npy",
    "**/*holdout*prob*.npy",
    "**/*holdout*y_prob*.npy",
    "**/*y_prob_test*.npy",
    "**/*test*prob*.npy",
])

if os.path.exists(split_path):
    split = json.load(open(split_path, "r"))
    test_idx = extract_test_idx(split)
    y_true_test = y_true_all[test_idx]
    print("✅ split:", split_path, "| test_idx:", test_idx.shape)

    if holdout_prob_path is None:
        print("⚠️ No HOLDOUT prob file found. (OK) We'll still compute CV/OOF + bootstrap.")
    else:
        y_prob_test = np.load(holdout_prob_path)
        print("✅ holdout prob:", y_prob_test.shape, "from", holdout_prob_path)

        m_holdout = compute_metrics(y_true_test, y_prob_test, thr=THR)
        print("\n📌 HOLDOUT METRICS")
        for k,v in m_holdout.items():
            print(f"{k:28s}: {v}")

        ci_holdout = bootstrap_ci(y_true_test, y_prob_test, thr=THR, n_boot=N_BOOT, seed=SEED)
        outp = os.path.join(OUT_DIR, f"holdout_bootstrap_ci_thr{THR}_B{N_BOOT}.json")
        json.dump(ci_holdout, open(outp, "w"), indent=2)
        print("✅ Saved:", outp)
else:
    print("⚠️ split_70_15_15.json not found. Skipping holdout section.")

# -----------------------
# CV / OOF metrics + bootstrap CI + fold metrics
# -----------------------
y_prob_oof, fold_ids, mask, oof_path, fold_ids_path = load_oof_and_fold_ids(RUN_A)
print("\n✅ OOF prob:", y_prob_oof.shape, "from", oof_path)
print("✅ fold_ids:", fold_ids.shape, "from", fold_ids_path)
print("✅ filled rows:", int(mask.sum()), "/", int(len(mask)))

y_true_oof = y_true_all[mask]
y_prob_oof_m = y_prob_oof[mask]

m_oof = compute_metrics(y_true_oof, y_prob_oof_m, thr=THR)
print("\n📌 CV OOF METRICS (filled rows)")
for k,v in m_oof.items():
    print(f"{k:28s}: {v}")

ci_oof = bootstrap_ci(y_true_oof, y_prob_oof_m, thr=THR, n_boot=N_BOOT, seed=SEED)
outp = os.path.join(OUT_DIR, f"cv_oof_bootstrap_ci_thr{THR}_B{N_BOOT}.json")
json.dump(ci_oof, open(outp, "w"), indent=2)
print("✅ Saved:", outp)

folds_A = fold_metrics_from_oof(y_true_all, y_prob_oof, fold_ids, mask, thr=THR)
outp = os.path.join(OUT_DIR, f"cv_fold_metrics_thr{THR}.json")
json.dump(folds_A, open(outp, "w"), indent=2)
print("✅ Saved:", outp)

# -----------------------
# PAIRED T-TEST (optional)
# -----------------------
if RUN_B is not None:
    y_true_B, _ = load_y_multi_hot(RUN_B, SOURCE_DIR)
    y_prob_oof_B, fold_ids_B, mask_B, _, _ = load_oof_and_fold_ids(RUN_B)

    # Ensure same ordering
    assert fold_ids_B.shape == fold_ids.shape, "fold_ids shape mismatch between runs"
    # recommended: use same mask positions (intersection)
    joint_mask = mask & mask_B

    folds_B = fold_metrics_from_oof(y_true_B, y_prob_oof_B, fold_ids_B, joint_mask, thr=THR)

    tests = {}
    for key in ["f1_macro", "precision_macro", "recall_macro", "roc_auc_macro", "auprc_macro"]:
        tests[key] = paired_ttest_from_folds(folds_A, folds_B, metric_key=key)

    outp = os.path.join(OUT_DIR, f"paired_ttest_vs_baseline_thr{THR}.json")
    json.dump(tests, open(outp, "w"), indent=2)
    print("✅ Saved:", outp)

print("\nDONE ✅  Results saved in:", OUT_DIR)

In [ ]:
from captum.attr import IntegratedGradients
import matplotlib.pyplot as plt
import numpy as np
import os, json, torch

def run_ig_examples(ckpt_path, n_examples=3, target_mode="first_true"):
    # load model
    K = 4
    model = (CNN2D_Transformer_STFT(n_classes=K, dropout_p=DROPOUT_P) if USE_STFT
             else HybridCNNTransformer_1D(n_classes=K, dropout_p=DROPOUT_P)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    ig = IntegratedGradients(model)

    with open(HOLDOUT_SPLIT_PATH, "r") as f:
        split = json.load(f)
    idxs = split["test"][:n_examples]

    ds = BeatsDataset(
        beats_path, Y_path, idxs,
        use_stft=USE_STFT,
        n_fft=STFT_NFFT, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, log1p=STFT_LOG1P
    )

    save_dir = os.path.join(NEW_RUN_DIR, "xai_ig")
    os.makedirs(save_dir, exist_ok=True)

    for j in range(len(ds)):
        X, ytrue = ds[j]                       # X: (B,12,1250)
        X = X.unsqueeze(0).to(device)          # (1,B,12,1250)

        ytrue_np = ytrue.numpy()
        true_labels = np.where(ytrue_np > 0.5)[0]

        if target_mode == "first_true" and len(true_labels) > 0:
            target = int(true_labels[0])
        else:
            with torch.no_grad():
                prob = torch.sigmoid(model(X)).squeeze(0).detach().cpu().numpy()
            target = int(np.argmax(prob))

        baseline = torch.zeros_like(X)
        attr = ig.attribute(X, baselines=baseline, target=target)   # same shape as X

        attr_np = attr.detach().cpu().numpy()[0]  # (B,12,1250)
        np.save(os.path.join(save_dir, f"ig_attr_{j}.npy"), attr_np)

        # Simple visualization: mean abs attribution over beats -> (12,1250)
        A = np.mean(np.abs(attr_np), axis=0)

        plt.figure()
        plt.imshow(A, aspect="auto")
        plt.title(f"IG | sample {j} | target={target}")
        plt.xlabel("time")
        plt.ylabel("lead")
        plt.colorbar()
        plt.tight_layout()
        plt.show()

        print(f"✅ IG saved for sample {j} -> {save_dir}")

    print("XAI dir:", save_dir)

best_ckpt = os.path.join(NEW_RUN_DIR, "report_holdout_70_15_15", "best_holdout_70_15_15.pt")
run_ig_examples(best_ckpt, n_examples=3)


## Note on external validation

The uploaded source notebook does not contain a single clean, self-contained PTB Diagnostic external-validation block comparable in quality to the PTB-XL pipeline cells kept here. For the repository artefact, it is better to keep PTB Diagnostic inference-only evaluation in a separate notebook or script once the code is cleaned in the same report-aligned style.